# Article text representations

BT4222 Group 6 · representations 1/3.

Fit **title TF-IDF + SVD on training news only**, freeze the transform, then encode title and abstract
in that same space. Export title, abstract and fused text vectors, article-row mappings and title token
sequences for the NRMS owner. This is a reproducible compact baseline, not a trained NRMS model.

**In Summary,**
- Pre-computed 64-dimensional vectors — title64, abstract64, and the fused text64 (0.7·title + 0.3·abstract, falling back to whichever field is valid) — are used for fast, large-scale candidate search and lightweight feature calculations in tabular ML models.
- Ordered word token sequences (title_token_ids + title_token_mask, with the training-only title_vocabulary.json where 0=PAD, 1=UNK) are fed into deep neural networks like NRMS to capture exact word context and accurately re-rank the top candidate recommendations.
- Two label-encoded columns (category_code/subcategory_code, training-only vocabularies with 0=unseen) serve as categorical features.
All arrays are joined to articles via article_index.parquet (row 0 is padding), and the frozen text_transform.pkl encodes new articles at serving time.



The core method is implemented in the cells below. Shared imports handle file loading, cache/hash checks, storage, and audit reports. Build calls explicitly receive the notebook-defined functions. `REBUILD=False` reuses saved datasets; set it to `True` after editing a method and rebuild dependent outputs.

In [1]:
from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

# Find the source package whether this notebook starts in the root or its subfolder.
CODE_ROOT = next(
    (
        p
        for p in (
            Path.cwd(),
            *Path.cwd().parents,
            Path("/content/drive/MyDrive/BT4222"),
        )
        if (p / "src/newsrec").is_dir()
    ),
    None,
)
if CODE_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook from the project checkout containing src/newsrec/."
    )
sys.path.insert(0, str(CODE_ROOT / "src"))
from newsrec.common import Settings, saved_stage_ready

DATA_ROOT = (
    None  # Optional: point to another folder holding the project's saved datasets.
)
ROOT = (
    Path(DATA_ROOT or os.environ.get("NEWSREC_ROOT") or CODE_ROOT)
    .expanduser()
    .resolve()
)
ARTIFACTS = ROOT / "3_Representations/artifacts"
print("Data/output root:", ROOT)

from newsrec.representations import build_news, representation_summary, text_neighbors
from collections import Counter
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from newsrec.common import DEFAULT_SETTINGS, normalise
from newsrec.representations import clean_text

Data/output root: /Users/bryan03/Desktop/BT4222-Group-6-


In [2]:
# Defaults live in src/newsrec/common.py; override with Settings(...) if needed.
SETTINGS = Settings()
REBUILD = False  # Reuse saved outputs; True regenerates this notebook's results.

## 1. Fit and export

Defaults: title unigrams/bigrams, up to 50,000 terms, `min_df=2`, SVD64, fixed seed.
All vectors are float32 and L2-normalised when valid. The combined view weights title/abstract 0.7/0.3;
a missing/OOV field falls back to the other valid field. Completely unsupported text remains zero with a mask.
Array row 0 is reserved for padding; real `news_code` values must go through `article_index.parquet`.
Training-fitted category/subcategory vocabularies reserve code 0 for unseen values.

Rebuilding representations invalidates dependent artifacts. Use one settings across the three notebooks.

### Training-only text fit and frozen transforms

The mask comes from `in_news_train`. Only training titles enter `fit_transform` or `svd.fit`; all catalogue titles and abstracts use the fitted models through `transform`. Row 0 is padding; missing/unsupported content stays masked.

**Scope:** training-only fitting prevents validation/test fitting leakage. This is an offline training-catalogue representation, not a text model refitted at each training impression timestamp.

In [3]:
def fit_text_space(news, cfg=DEFAULT_SETTINGS):
    """Fit vocabulary/IDF and SVD on training titles; freeze for every transform."""
    titles = news.title.map(clean_text)
    abstracts = news.abstract.map(clean_text)
    train = news.in_news_train.fillna(False).to_numpy(dtype=bool)
    if train.sum() < 2:
        raise ValueError("At least two training articles are required")
    tfidf = TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=cfg.max_features,
        min_df=cfg.min_df,
        dtype=np.float32,
    )
    train_tfidf = tfidf.fit_transform(titles[train])
    components = min(cfg.text_dim, train_tfidf.shape[0] - 1, train_tfidf.shape[1] - 1)
    if components < 1:
        raise ValueError(
            "Insufficient training vocabulary for SVD; inspect min_df/text"
        )
    svd = TruncatedSVD(n_components=components, random_state=cfg.seed)
    svd.fit(train_tfidf)
    matrices = []
    for texts in (titles, abstracts):
        reduced = normalise(svd.transform(tfidf.transform(texts)))
        dense = np.zeros((len(titles) + 1, cfg.text_dim), np.float32)
        dense[1:, :components] = reduced
        matrices.append(dense)
    title, abstract = matrices
    tv = np.linalg.norm(title, axis=1) > 0
    av = np.linalg.norm(abstract, axis=1) > 0
    text = normalise(cfg.title_weight * title + (1 - cfg.title_weight) * abstract)
    text[tv & ~av] = title[tv & ~av]
    text[av & ~tv] = abstract[av & ~tv]
    return tfidf, svd, components, title, abstract, text, tv, av

### Training vocabulary for NRMS and categories

Token/category vocabularies also use only training articles; unseen tokens get ID 1 and unseen categories ID 0.

In [4]:
def encode_title_tokens(titles, train, cfg=DEFAULT_SETTINGS):
    tokenise = TfidfVectorizer().build_analyzer()
    frequency = Counter(token for text_ in titles[train] for token in tokenise(text_))
    token_vocab = {
        w: i + 2
        for i, (w, _) in enumerate(
            sorted(frequency.items(), key=lambda x: (-x[1], x[0]))[: cfg.max_features]
        )
    }
    token_ids = np.zeros((len(titles) + 1, cfg.title_tokens), np.int32)
    token_lengths, unknown, total = ([], 0, 0)
    for i, text_ in enumerate(titles, 1):
        tokens = tokenise(text_)
        token_lengths.append(len(tokens))
        ids = [token_vocab.get(w, 1) for w in tokens]
        unknown += sum(x == 1 for x in ids)
        total += len(ids)
        token_ids[i, : min(len(ids), cfg.title_tokens)] = ids[: cfg.title_tokens]
    return token_ids, token_vocab, token_lengths, unknown, total


def fit_category_encodings(news, train):
    encodings = {}
    for col in ("category", "subcategory"):
        vocabulary = sorted(set(news.loc[train, col].dropna().astype(str)))
        lookup = {v: i + 1 for i, v in enumerate(vocabulary)}
        codes = news[col].map(lookup).fillna(0).astype(np.int32)
        encodings[col] = (codes, lookup)
    return encodings

In [5]:
if REBUILD or not saved_stage_ready(ROOT, SETTINGS, "news"):
    coverage = build_news(
        ROOT,
        SETTINGS,
        text_encoder=fit_text_space,
        token_encoder=encode_title_tokens,
        category_encoder=fit_category_encodings,
    )
else:
    print("Loaded saved article text representations.")
    report = pd.read_csv(ROOT / "results/representation_coverage.csv")
    coverage = report[report.stage.eq("text")]
display(coverage)

Loaded saved article text representations.


,split,representation,cohort,denominator,valid,fallback,valid_fraction,stage
0,train,title,split_catalogue,98671,98667,4,0.999959,text
1,train,abstract,split_catalogue,98671,93360,5311,0.946175,text
2,train,text,split_catalogue,98671,98671,0,1.000000,text
3,validation,title,split_catalogue,71991,71989,2,0.999972,text
4,validation,abstract,split_catalogue,71991,68340,3651,0.949285,text
5,validation,text,split_catalogue,71991,71991,0,1.000000,text
6,test,title,split_catalogue,72023,72022,1,0.999986,text
7,test,abstract,split_catalogue,72023,68359,3664,0.949127,text
8,test,text,split_catalogue,72023,72023,0,1.000000,text


## 2. Check dimensions, norms and quality

A present title can become a zero vector after train-vocabulary filtering. Abstract absence and zero-vector
rates are distinct. On a tiny fixture the effective SVD rank can be smaller than the configured dimension;
unused dimensions are padded with zeros and the effective rank is recorded.

In [6]:
display(representation_summary(ROOT, SETTINGS))
index = pd.read_parquet(ARTIFACTS / "article_index.parquet")
display(
    index[
        [
            "news_code",
            "title",
            "has_abstract",
            "has_title_vector",
            "has_abstract_vector",
        ]
    ].head()
)
meta = json.loads((ARTIFACTS / "news_manifest.json").read_text())
display(
    {
        k: meta[k]
        for k in [
            "effective_svd_dim",
            "fit_scope",
            "token_oov_fraction",
            "title_truncation_fraction",
        ]
    }
)

,representation,shape,MB,valid_articles
0,title64,"(104152, 64)",26.662912,104147
1,abstract64,"(104152, 64)",26.662912,98522
2,text64,"(104152, 64)",26.662912,104151


,news_code,title,has_abstract,has_title_vector,has_abstract_vector
0,1,Texans defensive tackle D.J. Reader is taking ...,True,True,True
1,2,Mormons to the Rescue?,True,True,True
2,4,"AP Top 25 poll ballot: Notre Dame, Oklahoma pl...",True,True,True
3,5,"Subway preacher, 79, needs 30 stitches after c...",True,True,True
4,6,Coco Gauff beats Jelena Ostapenko to claim fir...,True,True,True


{'effective_svd_dim': 64,
 'fit_scope': 'unique training-release article titles only',
 'token_oov_fraction': 0.0009094349794832558,
 'title_truncation_fraction': 0.0002784418776583998}

## 3. Inspect neighbours without using click labels

This is a qualitative sanity check of shared text coordinates, not evidence of a ranking gain.
The frozen vocabulary can leave new articles unsupported; the serving pipeline retains those articles
and uses entity/popularity/freshness fallback rather than dropping them.

In [7]:
display(text_neighbors(ROOT, count=5))
display(pd.read_csv(ROOT / "results/representation_cost.csv").tail())

,article_row,news_code,title,cosine,is_query
0,1,1,Texans defensive tackle D.J. Reader is taking ...,1.000000,True
1,31093,38640,Nick Offerman Is Getting His Own Scotch Courte...,0.890176,False
2,98576,123363,Nick Offerman Is Getting His Own Scotch Courte...,0.883896,False
3,77509,96916,Giants Hall of Famer Michael Strahan is a dead...,0.873511,False
4,23085,28619,Sheriff Gregory Tony isn't officially a candid...,0.869434,False
5,48903,61051,Jared McCann is answering a lot of big questions,0.860839,False


,stage,wall_seconds,sampled_peak_process_rss_mb,platform,cpu,logical_cpus,ram_gb
0,text,6.809420,1249.378304,macOS-26.6.2-arm64-arm-64bit,arm,10,17.179869
1,entity,3.183998,910.753792,macOS-26.6.2-arm64-arm-64bit,arm,10,17.179869
2,availability,0.808571,1673.281536,macOS-26.6.2-arm64-arm-64bit,arm,10,17.179869
3,supplied_profiles,27.588941,3564.404736,macOS-26.6.2-arm64-arm-64bit,arm,10,17.179869
4,ranking_features,9787.377177,4266.852352,macOS-26.6.2-arm64-arm-64bit,arm,10,17.179869


## Handoff

Consumers can load the saved arrays with NumPy and the article index with pandas after all three notebooks finish.
`title64` reproduces the report's multi-interest space; `text64` is a separate combined-content view.
NRMS receives `title_token_ids.npy`, its mask, and the training-only vocabulary (`0=PAD`, `1=UNK`);
its attention encoders and learned embeddings belong to the NRMS notebook.